# Downloading Xenium data

`spatialrefinery.download_xenium_study` fetches a 10x Genomics Xenium study's raw asset bundle from a manifest of `curl -O <url>` lines -- the same manifest 10x's dataset pages let you copy when you choose the "curl" download option. It retries transient failures, writes atomically (an interrupted download never leaves a truncated file behind), downloads in parallel, and unzips `*_outs.zip` archives in place (while always skipping the large `*_xe_outs.zip` Xenium Explorer bundles, which aren't needed for SpatialData conversion).

**What you need:** a manifest file for the study you want. **Runtime:** the dry run below is instant; the real download depends on your network and how many assets you select.

In [ ]:
from pathlib import Path

from spatialrefinery import download_xenium_study

## The manifest format

A manifest is a plain text file with one `curl -O <url>` line per asset. 10x's dataset pages (`10xgenomics.com/datasets/...`) offer this as a download option next to the individual file links -- copy it into a `.txt` file and pass that path as `source`.

The cell below writes a small example manifest so this notebook runs standalone. Replace the URLs with the ones from your own study before running for real.

In [ ]:
DATA_DIR = Path("xenium_example")
DATA_DIR.mkdir(exist_ok=True)

manifest_path = DATA_DIR / "manifest.txt"
manifest_path.write_text(
    "curl -O https://cf.10xgenomics.com/samples/xenium/0.0.0/example_study/example_study_outs.zip\n"
    "curl -O https://cf.10xgenomics.com/samples/xenium/0.0.0/example_study/example_study_he_image.ome.tif\n"
    "curl -O https://cf.10xgenomics.com/samples/xenium/0.0.0/example_study/example_study_he_imagealignment.csv\n"
)
print(manifest_path.read_text())

## Dry run: see what would be downloaded

`dry_run=True` resolves the manifest and reports the plan -- study names, asset kinds, destinations -- without any network activity. Useful to sanity-check a manifest before committing to a multi-GB download.

In [ ]:
plan = download_xenium_study(manifest_path, outdir=DATA_DIR / "raw", dry_run=True)
for result in plan:
    print(f"{result.asset.study:20s} {result.asset.kind:20s} -> {result.path}")

Each entry is a `DownloadResult`: `.status` (`"downloaded"`, `"cached"`, `"skipped"`, or `"failed"`), `.ok` (a bool convenience property), `.asset` (the source `RemoteAsset`, with `.url`, `.study`, `.filename`, `.kind`), and `.error` when a download failed.

## Downloading a subset

Restrict to specific asset `kinds` to avoid pulling everything -- here, just the main `"outs"` bundle (transcripts, cell/nucleus boundaries, morphology image) and skip the larger auxiliary images:

In [ ]:
raw_dir = DATA_DIR / "raw"
results = download_xenium_study(
    manifest_path,
    outdir=raw_dir,
    kinds=["outs"],
    max_workers=8,
)

failed = [r for r in results if not r.ok]
if failed:
    print(f"{len(failed)}/{len(results)} asset(s) failed:")
    for r in failed:
        print(f"  - {r.asset.url}: {r.error}")
else:
    print(f"Downloaded {len(results)} asset(s) to {raw_dir}")

## Output layout

Assets land under `outdir/<study>/<filename>`, with `*_outs.zip` unzipped in place (so you get a directory of raw Xenium files ready for the next tutorial):

```text
raw/
└── example_study/
    ├── example_study_outs.zip
    └── example_study_outs/
        ├── experiment.xenium
        ├── transcripts.parquet
        ├── cells.parquet
        ├── cell_boundaries.parquet
        ├── nucleus_boundaries.parquet
        └── morphology_focus/
```

## Running this as a script

```bash
python scripts/xenium_download.py --input_file manifest.txt --outdir raw_files --workers 8
```

## What's next

Continue to [Xenium to SpatialData zarr](xenium_to_zarr) to convert the downloaded bundle into a SpatialData zarr store.